# 3. Resolver e reconhecer o limite do solver

In [1]:
from pathlib import Path
import sys, json, numpy as np
ROOT = Path.cwd().resolve()
if not (ROOT / 'optimization_examples').is_dir():
    ROOT = ROOT.parent  # notebooks opened from codes/notebooks/
if not (ROOT / 'optimization_examples').is_dir():
    raise RuntimeError('Abra o notebook a partir do repositório codes ou codes/notebooks.')
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'results' / 'sequence'
OUT.mkdir(parents=True, exist_ok=True)
def save(name, value):
    (OUT / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


PuLP é a interface de modelagem; CBC é o solver aberto. O exemplo usa PuLP 3.3.x. Não interpretar apenas LpStatus: observar também sol_status, log e viabilidade. O limite de tempo não é prova de inviabilidade.

In [2]:
import pulp as pl
from optimization_examples.selection import small_instance, stress_instance, solve
print('PuLP:', pl.__version__, 'Solvers:', pl.listSolvers(onlyAvailable=True))
small = solve(small_instance(), log_path=OUT/'small-cbc.log')
print(small['status'], 'prioridade:', small['objective'], 'seleção:', small['x'])
save('small.json', small)

PuLP: 3.3.2 Solvers: ['PULP_CBC_CMD']
Ótimo certificado prioridade: 36.0 seleção: [0.0, 1.0, 0.0, 1.0, 1.0, 1.0]


A instância de estresse tem 250 kits e dez recursos artificiais. Ela preserva o modelo, amplia seu tamanho e impõe 0,3 s ao CBC. É demonstração computacional, não descrição de uma operação real. Pode encerrar de outra forma em outra máquina.

In [3]:
large = solve(stress_instance(), time_limit=.3, log_path=OUT/'stress-cbc.log')
print('Status verificado:', large['status'])
print('Status PuLP:', large['solver_status'], '/', large['solution_status'])
print('Prioridade viável:', large['objective'], 'tempo medido:', large['seconds'])
for line in large['log_summary']: print(line)
save('stress.json', large)

Status verificado: Viável; ótimo não certificado
Status PuLP: Optimal / Solution Found
Prioridade viável: 5646.0 tempo medido: 0.3325
Result - Stopped on time limit
Objective value:                5646.00000000
Upper bound:                    5679.957
Gap:                            -0.01
Time (Wallclock seconds):       0.30
